# UN General Debate NLP — Africa extension

Professional reconstruction of the **Social Media & Web Analytics (SMWA)** extension. It uses the same UN General Debate corpus as the AI-course project but focuses on African speeches and adds sentiment, LDA, NMF, BERTopic, and network analysis.

**Interpretation boundary:** this is descriptive text analysis. Topic prevalence, sentiment, and network centrality are not causal effects.


## 1. Setup

Install the classical stack with `pip install -r requirements.txt`, then run `python scripts/bootstrap_nltk.py`. BERTopic is optional: `pip install -r requirements-bertopic.txt`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import add_text_features, filter_africa, load_ungd
from src.sentiment import add_sentiment_features
from src.topic_models import (
    bertopic_word_lists, fit_bertopic, fit_lda, fit_nmf,
    lda_topic_prevalence, tokenize_for_coherence, topic_coherence,
)
from src.visualization import (
    make_wordcloud, plot_sentiment_over_time,
    plot_speeches_over_time, plot_word_count_distribution,
)

DATA_PATH = ROOT / "data" / "un-general-debates.csv"


## 2. Construct the Africa sample

The final computational coursework snapshot contains **7,507 speeches (1970–2015)**, of which **2,159** are from the 54 African country codes used in the project.


In [ ]:
ungd = load_ungd(DATA_PATH)
africa = add_text_features(filter_africa(ungd))

summary = pd.Series({
    "full_corpus_speeches": len(ungd),
    "africa_speeches": len(africa),
    "africa_country_codes": africa["country"].nunique(),
    "first_year": int(africa["year"].min()),
    "last_year": int(africa["year"].max()),
    "median_words": float(africa["word_count"].median()),
})
summary


## 3. Exploratory analysis


In [ ]:
plot_speeches_over_time(africa)
plot_word_count_distribution(africa, title="African UNGD speech-length distribution")
make_wordcloud(africa["processed_text"], title="Africa: high-frequency diplomatic vocabulary")


The submitted SMWA paper describes a right-skewed length distribution, relatively consistent session participation, and high-frequency vocabulary including *United Nations*, *international community*, *developing country*, and *Security Council*. The original figures are preserved under `coursework/social_media_web_analytics/`.


## 4. Sentiment — corrected sentence-level VADER

The coursework contains a sentiment implementation that tokenized too early. The professional rebuild scores sentence-like units from the **original speech text** and aggregates sentence-level VADER compound scores to the speech level.


In [ ]:
africa_sentiment = add_sentiment_features(africa, text_column="text")
plot_sentiment_over_time(
    africa_sentiment,
    title="Africa: mean sentence-level VADER sentiment by year",
)
africa_sentiment["sentiment_label"].value_counts(normalize=True)


The submitted paper reported a gradual upward sentiment trend and associated positive language with peace/justice/hope and negative language with war/conflict/terrorism/poverty/violence. Those remain **coursework-reported findings**; this cell defines the corrected measurement used for future reproduction.


## 5. LDA


In [ ]:
documents = africa["processed_text"].tolist()
lda = fit_lda(documents, num_topics=10, passes=50, random_state=0)

lda_topics = {
    topic_id: [word for word, _ in lda.model.show_topic(topic_id, topn=10)]
    for topic_id in range(lda.model.num_topics)
}
lda_prevalence = pd.DataFrame({
    "topic": range(lda.model.num_topics),
    "mean_probability": lda_topic_prevalence(lda),
    "top_words": [", ".join(lda_topics[i]) for i in range(lda.model.num_topics)],
}).sort_values("mean_probability", ascending=False)

print(f"LDA c_v coherence (recomputed with professional pipeline): {lda.coherence:.4f}")
lda_prevalence


## 6. NMF


In [ ]:
nmf = fit_nmf(documents, num_topics=10, random_state=0)
print(f"NMF c_v coherence (recomputed with professional pipeline): {nmf.coherence:.4f}")

pd.DataFrame({
    "topic": range(len(nmf.topics)),
    "top_words": [", ".join(words) for words in nmf.topics],
})


## 7. BERTopic (optional)

The original SMWA analysis also estimated BERTopic. It is intentionally optional because it downloads a sentence-transformer model and has a much heavier dependency stack.


In [ ]:
RUN_BERTOPIC = False

if RUN_BERTOPIC:
    bertopic_model, assignments, probabilities = fit_bertopic(
        documents, n_gram_range=(1, 3), min_topic_size=10
    )
    bertopic_topics = bertopic_word_lists(bertopic_model, top_n=10)
    reference_tokens = tokenize_for_coherence(documents)
    bertopic_cv = topic_coherence(bertopic_topics, reference_tokens)
    print(f"BERTopic c_v coherence on common reference corpus: {bertopic_cv:.4f}")
else:
    print("BERTopic skipped. Install requirements-bertopic.txt and set RUN_BERTOPIC=True.")


### Coursework-reported coherence

The submitted paper reported **LDA 0.3663, NMF 0.5464, BERTopic 0.7768** and interpreted BERTopic as the most coherent. These values are preserved for provenance, but the original three coherence sections did **not** construct their evaluation inputs identically. They should therefore not be treated as a strict apples-to-apples model benchmark.


## 8. Network analysis

The country-mention network is reconstructed separately in [`03_africa_network_extension.ipynb`](03_africa_network_extension.ipynb), where the original ISO-code matching rule and the professional name/alias matching rule are explicitly distinguished.


## 9. What this notebook establishes

- reproducible Africa sample construction;
- clear EDA and word-cloud generation;
- corrected sentence-level sentiment measurement;
- executable LDA and NMF;
- optional BERTopic with harmonized coherence evaluation;
- explicit separation of coursework-reported findings from professional re-estimation.

See `docs/methodology.md`, `docs/findings.md`, and `docs/qa.md` for the full audit.
